In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Sensitivity scenarios (baseline damage rates)
BASELINES = [0.02, 0.10]   # keep 0.05 as the existing v1b file

# --- resolve actual repo root (notebooks/ -> repo root) ---
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / "src").exists(), f"Expected src/ at repo root: {REPO_ROOT}"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

from src.rtm.io_hazard import load_rtm_pluvial_v1_buildings

OUT_DIR = REPO_ROOT / "outputs" / "rtm" / "outcomes"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outcome outputs:", OUT_DIR)

Repo root: C:\Users\C.Price\Habnetic\resilient-housing-bayes
Outcome outputs: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes


In [2]:
E_PATH = REPO_ROOT / "outputs" / "rtm" / "water_exposure_Ehat_v0.parquet"
if not E_PATH.exists():
    raise FileNotFoundError(f"Missing E_hat parquet at {E_PATH}")

E = pd.read_parquet(E_PATH)

if "bldg_id" not in E.columns:
    raise ValueError(f"E_hat table missing bldg_id. Columns: {list(E.columns)}")

if "E_hat" not in E.columns:
    # try common alternates
    candidates = [c for c in E.columns if c.lower() in {"ehat", "e_hat", "water_exposure_ehat", "exposure"}]
    if candidates:
        E = E.rename(columns={candidates[0]: "E_hat"})
    else:
        raise ValueError(f"Could not find E_hat column. Columns: {list(E.columns)}")

E = E[["bldg_id", "E_hat"]].copy()
print("E rows:", len(E), "unique bldg_id:", E["bldg_id"].is_unique)
E.head()

E rows: 221324 unique bldg_id: True


,bldg_id,E_hat
0,305012,0.536838
1,313960,0.677579
2,313263,0.251841
3,310491,0.189019
4,313127,-0.292821


In [3]:
haz = load_rtm_pluvial_v1_buildings()
print("Haz rows:", len(haz), "unique bldg_id:", haz["bldg_id"].is_unique)
haz.head()

Haz rows: 221324 unique bldg_id: True


,bldg_id,H_pluvial_v1_mm
0,305012,25.422161
1,313960,25.418823
2,313263,25.423113
3,310491,25.424500
4,313127,25.423491


In [4]:
df = E.merge(haz, on="bldg_id", how="inner", validate="one_to_one")
print("Joined rows:", len(df))
print("Drops vs E:", len(E) - len(df))
print("Drops vs hazard:", len(haz) - len(df))

# Basic sanity
assert len(df) == 221_324, "Unexpected join row count; investigate before generating outcome."
assert df["E_hat"].notna().all()
assert df["H_pluvial_v1_mm"].notna().all()

df.head()

Joined rows: 221324
Drops vs E: 0
Drops vs hazard: 0


,bldg_id,E_hat,H_pluvial_v1_mm
0,305012,0.536838,25.422161
1,313960,0.677579,25.418823
2,313263,0.251841,25.423113
3,310491,0.189019,25.424500
4,313127,-0.292821,25.423491


In [5]:
def zscore(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    mu = x.mean()
    sd = x.std(ddof=1)
    if sd <= 0:
        raise ValueError("Standard deviation is zero; cannot z-score.")
    return (x - mu) / sd

df["E_std"] = zscore(df["E_hat"])
df["H_std"] = zscore(df["H_pluvial_v1_mm"])

print("E_std mean/std:", float(df["E_std"].mean()), float(df["E_std"].std(ddof=1)))
print("H_std mean/std:", float(df["H_std"].mean()), float(df["H_std"].std(ddof=1)))

E_std mean/std: 4.4945863533287865e-19 0.9999999999999999
H_std mean/std: 1.561788497311158e-15 0.9999999999999999


In [6]:
seed = 42
rng = np.random.default_rng(seed)

beta_E = 1.0
beta_H = 0.6

def logit(p: float) -> float:
    return float(np.log(p / (1.0 - p)))

for p0 in BASELINES:
    alpha = logit(p0)

    linpred = alpha + beta_E * df["E_std"].values + beta_H * df["H_std"].values
    p_true = 1.0 / (1.0 + np.exp(-linpred))

    Y = rng.binomial(n=1, p=p_true, size=len(df)).astype(np.int8)

    col_y = f"Y_damage_v1b_p{int(round(p0*100)):02d}"
    col_p = f"p_true_p{int(round(p0*100)):02d}"

    df[col_p] = p_true.astype(np.float32)
    df[col_y] = Y

    print(f"\nScenario p0={p0:.2f}")
    print("Realized damage rate:", float(df[col_y].mean()))
    print("p_true min/median/p95/max:",
          float(np.min(p_true)),
          float(np.median(p_true)),
          float(np.quantile(p_true, 0.95)),
          float(np.max(p_true)))


Scenario p0=0.02
Realized damage rate: 0.03378756935533426
p_true min/median/p95/max: 0.0003251992969588295 0.023249415140552544 0.09506395595040168 0.5231506786781827

Scenario p0=0.10
Realized damage rate: 0.14319730350075002
p_true min/median/p95/max: 0.0017679741991464988 0.11472548094643076 0.3638439876587551 0.8565915926361671


In [7]:
for p0 in BASELINES:
    tag = f"p{int(round(p0*100)):02d}"
    col_y = f"Y_damage_v1b_{tag}"
    col_p = f"p_true_{tag}"

    out = df[["bldg_id", col_y, col_p]].copy()
    out = out.rename(columns={col_y: "Y_damage", col_p: "p_true"})

    out["outcome_src"] = "synthetic"
    out["outcome_version"] = f"v1b_{tag}"
    out["seed"] = seed
    out["p0_baseline"] = float(p0)
    out["beta_E"] = beta_E
    out["beta_H"] = beta_H
    out["alpha"] = float(np.log(p0 / (1.0 - p0)))

    out_path = OUT_DIR / f"Y_damage_v1b_{tag}.parquet"
    out.to_parquet(out_path, index=False)

    print("Saved:", out_path)
    print("Rows:", len(out))
    print(out.head())

Saved: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1b_p02.parquet
Rows: 221324
   bldg_id  Y_damage    p_true outcome_src outcome_version  seed  p0_baseline  \
0   305012         0  0.017803   synthetic         v1b_p02    42         0.02   
1   313960         0  0.020772   synthetic         v1b_p02    42         0.02   
2   313263         0  0.012784   synthetic         v1b_p02    42         0.02   
3   310491         0  0.011925   synthetic         v1b_p02    42         0.02   
4   313127         0  0.006735   synthetic         v1b_p02    42         0.02   

   beta_E  beta_H    alpha  
0     1.0     0.6 -3.89182  
1     1.0     0.6 -3.89182  
2     1.0     0.6 -3.89182  
3     1.0     0.6 -3.89182  
4     1.0     0.6 -3.89182  
Saved: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1b_p10.parquet
Rows: 221324
   bldg_id  Y_damage    p_true outcome_src outcome_version  seed  p0_baseline  \
0   305012         1  0.08

In [8]:
for p0 in BASELINES:
    tag = f"p{int(round(p0*100)):02d}"
    path = OUT_DIR / f"Y_damage_v1b_{tag}.parquet"
    tmp = pd.read_parquet(path)

    assert tmp["bldg_id"].is_unique
    assert set(tmp["Y_damage"].unique()).issubset({0, 1})
    assert tmp["Y_damage"].isna().sum() == 0
    assert tmp["p_true"].isna().sum() == 0

    print(f"QA passed for {path.name}. Realized rate={tmp['Y_damage'].mean():.4f}")

QA passed for Y_damage_v1b_p02.parquet. Realized rate=0.0338
QA passed for Y_damage_v1b_p10.parquet. Realized rate=0.1432
